# Figure 10

With this notebook, we generate Figure 10 in Ronchi et al. (2026). 

Note that in order to make this notebook work, you first need to download the results data from `/data/magnesia/common/paper_ronchi_etal_2025/B_double_lognorm_dip-tor_heavy/experiments_paper.zip`, copy it into the folder `ML-Poppyns/data/paper_results/ronchi_etal_2026` and unpack the file so that the results data will be saved in the folder `ML-Poppyns/data/paper_results/ronchi_etal_2026/experiments_paper`.

In [ ]:
# Import libraries.
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pathlib
import os

import utilities.plot_settings

from matplotlib.ticker import FuncFormatter

formatter = FuncFormatter(lambda y, _: "{:.16g}".format(y))

import matplotlib.lines as mlines
from scipy.stats import gaussian_kde

In [ ]:
root_path = "../../data/paper_results/ronchi_etal_2026/experiments_paper"

simulations_path = (
    f"{root_path}/best_simulations_B_double_lognormal_dip-tor_heavy_youngxdins"
)

# Number of samples in the parsed directory.
n_sim = len(next(os.walk(simulations_path))[1])

In [ ]:
P_radio_sim_list = []
chi_radio_sim_list = []

P_pmps_sim_list = []
chi_pmps_sim_list = []

P_smps_sim_list = []
chi_smps_sim_list = []

P_htru_sim_list = []
chi_htru_sim_list = []

P_x_sim_list = []
chi_x_sim_list = []

P_xdins_sim_list = []
chi_xdins_sim_list = []

P_x_young_sim_list = []
chi_x_young_sim_list = []

In [ ]:
# Load simulation data and save the relevant parameters in lists.
# In particular, we need the spin period, P, and the inclination angle, chi.
for i in range(n_sim):
    path_to_simulation = f"{simulations_path}/{i:06d}"

    config_json = json.load(
        open(
            pathlib.Path().joinpath(path_to_simulation, "configuration.json"),
        )
    )

    # Skip simulations where the birth rate exceeded the maximum allowed to not bias the birth rate estimate.
    if (
        (config_json["birth_rate_PMPS_at_match"] == 0)
        | (config_json["birth_rate_SMPS_at_match"] == 0)
        | (config_json["birth_rate_HTRU_low_mid_at_match"] == 0)
        | (config_json["birth_rate_xray_realistic_at_match"] == 0)
    ):
        continue

    # Load the `.pkl.gz` files containing the survey results to import.
    df_PMPS_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_PMPS_results.pkl.gz"
        ),
        compression="gzip",
    )
    df_PMPS_sim.head()

    df_SMPS_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_SMPS_results.pkl.gz"
        ),
        compression="gzip",
    )

    df_HTRU_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_HTRU_low_mid_results.pkl.gz"
        ),
        compression="gzip",
    )
    df_x_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_xray_realistic_results.pkl.gz"
        ),
        compression="gzip",
    )

    # Extract relevant quantities.
    P_pmps_sim = df_PMPS_sim["P"]["[s]"].to_numpy()
    chi_pmps_sim = df_PMPS_sim["chi"]["[rad]"].to_numpy()

    P_smps_sim = df_SMPS_sim["P"]["[s]"].to_numpy()
    chi_smps_sim = df_SMPS_sim["chi"]["[rad]"].to_numpy()

    P_htru_sim = df_HTRU_sim["P"]["[s]"].to_numpy()
    chi_htru_sim = df_HTRU_sim["chi"]["[rad]"].to_numpy()

    # Remove duplicates, i.e., same pulsars detected with different radio surveys.
    unique_vals, idx = np.unique(
        np.concatenate((P_pmps_sim, P_smps_sim, P_htru_sim)), return_index=True
    )
    P_radio_sim = unique_vals[np.argsort(idx)]
    unique_vals, idx = np.unique(
        np.concatenate((chi_pmps_sim, chi_smps_sim, chi_htru_sim)),
        return_index=True,
    )
    chi_radio_sim = unique_vals[np.argsort(idx)]

    P_x_sim = df_x_sim["P"]["[s]"].to_numpy()
    chi_x_sim = df_x_sim["chi"]["[rad]"].to_numpy()
    d_x_sim = df_x_sim["dist"]["[kpc]"].to_numpy()
    age_x_sim = df_x_sim["age"]["[yr]"].to_numpy()

    # Filter young magnetars.
    young_mask = age_x_sim <= 2.0e3

    P_x_young_sim = P_x_sim[young_mask]
    chi_x_young_sim = chi_x_sim[young_mask]

    # Filter X-ray emitting NSs with XDINS-like properties.
    xdins_mask = (d_x_sim <= 0.5) & (age_x_sim >= 1.0e5)

    P_xdins_sim = P_x_sim[xdins_mask]
    chi_xdins_sim = chi_x_sim[xdins_mask]

    # Append the values to the corresponding lists.
    P_radio_sim_list.append(P_radio_sim)
    chi_radio_sim_list.append(chi_radio_sim)

    P_x_sim_list.append(P_x_sim)
    chi_x_sim_list.append(chi_x_sim)

    P_x_young_sim_list.append(P_x_young_sim)
    chi_x_young_sim_list.append(chi_x_young_sim)

    P_xdins_sim_list.append(P_xdins_sim)
    chi_xdins_sim_list.append(chi_xdins_sim)

In [ ]:
P_radio_sim_all = np.concatenate(P_radio_sim_list)
chi_radio_sim_all = np.concatenate(chi_radio_sim_list)

P_x_sim_all = np.concatenate(P_x_sim_list)
chi_x_sim_all = np.concatenate(chi_x_sim_list)

P_x_young_sim_all = np.concatenate(P_x_young_sim_list)
chi_x_young_sim_all = np.concatenate(chi_x_young_sim_list)

P_xdins_sim_all = np.concatenate(P_xdins_sim_list)
chi_xdins_sim_all = np.concatenate(chi_xdins_sim_list)

## Plot inclination angle vs spin period

In [ ]:
def density_contour(x, y, ax, **kwargs):
    """
    Helper function to make a density contour plot from a point cloud.
    """
    xy = np.vstack([x, y])
    kde = gaussian_kde(xy)
    xi, yi = np.meshgrid(
        np.linspace(np.min(x), np.max(x), 200),
        np.linspace(np.min(y), np.max(y), 200),
    )
    zi = kde(np.vstack([xi.flatten(), yi.flatten()]))
    zi = zi.reshape(xi.shape)

    return ax.contour(10**xi, yi, zi, **kwargs)


def density_contourf(x, y, ax, threshold=0.05, **kwargs):
    """
    Helper function to make a density plot from a point cloud.
    """
    xy = np.vstack([x, y])
    kde = gaussian_kde(xy)

    xi, yi = np.meshgrid(
        np.linspace(np.min(x), np.max(x), 200),
        np.linspace(np.min(y), np.max(y), 200),
    )

    zi = kde(np.vstack([xi.ravel(), yi.ravel()])).reshape(xi.shape)

    # Mask low-density background.
    zi = np.ma.masked_less(zi, threshold * zi.max())

    return ax.contourf(10**xi, yi, zi, **kwargs)

In [ ]:
quantile_chi_radio = np.quantile(
    chi_radio_sim_all / np.pi * 180, [0.16, 0.5, 0.84]
)
median_chi_radio = quantile_chi_radio[1]
lower_chi_radio = median_chi_radio - quantile_chi_radio[0]
upper_chi_radio = quantile_chi_radio[2] - median_chi_radio

quantile_chi_x = np.quantile(chi_x_sim_all / np.pi * 180, [0.16, 0.5, 0.84])
median_chi_x = quantile_chi_x[1]
lower_chi_x = median_chi_x - quantile_chi_x[0]
upper_chi_x = quantile_chi_x[2] - median_chi_x

quantile_chi_x_young = np.quantile(
    chi_x_young_sim_all / np.pi * 180, [0.16, 0.5, 0.84]
)
median_chi_x_young = quantile_chi_x_young[1]
lower_chi_x_young = median_chi_x_young - quantile_chi_x_young[0]
upper_chi_x_young = quantile_chi_x_young[2] - median_chi_x_young

quantile_chi_xdins = np.quantile(
    chi_xdins_sim_all / np.pi * 180, [0.16, 0.5, 0.84]
)
median_chi_xdins = quantile_chi_xdins[1]
lower_chi_xdins = median_chi_xdins - quantile_chi_xdins[0]
upper_chi_xdins = quantile_chi_xdins[2] - median_chi_xdins

print(
    f"inclination angle radio pulsars: {median_chi_radio} + {upper_chi_radio} - {lower_chi_radio}"
)
print(
    f"inclination angle X-ray detected neutron stars: {median_chi_x} + {upper_chi_x} - {lower_chi_x}"
)
print(
    f"inclination angle young magnetars: {median_chi_x_young} + {upper_chi_x_young} - {lower_chi_x_young}"
)
print(
    f"inclination angle XDINS-like: {median_chi_xdins} + {upper_chi_xdins} - {lower_chi_xdins}"
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xlabel(r"Spin period $P$ [s]")
ax.set_ylabel(r"Inclination angle $\chi$ [deg]")
ax.set_xscale("log")
ax.set_xlim(0.05, 200)
# ax.set_yscale("log")
ax.xaxis.set_major_formatter(formatter)

density_contourf(
    np.log10(P_radio_sim_all),
    chi_radio_sim_all / np.pi * 180,
    ax,
    threshold=0.1,
    levels=6,
    cmap="Greys",
    alpha=0.4,
)
density_contourf(
    np.log10(P_x_sim_all),
    chi_x_sim_all / np.pi * 180,
    ax,
    threshold=0.04,
    levels=6,
    cmap="Purples",
    alpha=0.5,
)
density_contourf(
    np.log10(P_xdins_sim_all),
    chi_xdins_sim_all / np.pi * 180,
    ax,
    threshold=0.1,
    levels=3,
    cmap="Oranges",
    alpha=0.3,
)
density_contourf(
    np.log10(P_x_young_sim_all),
    chi_x_young_sim_all / np.pi * 180,
    ax,
    threshold=0.1,
    levels=3,
    cmap="RdPu",
    alpha=0.2,
)

density_contour(
    np.log10(P_radio_sim_all),
    chi_radio_sim_all / np.pi * 180,
    ax,
    colors="tab:gray",
    levels=6,
    linewidths=3,
)
density_contour(
    np.log10(P_x_sim_all),
    chi_x_sim_all / np.pi * 180,
    ax,
    colors="tab:purple",
    levels=6,
    linewidths=3,
)
density_contour(
    np.log10(P_xdins_sim_all),
    chi_xdins_sim_all / np.pi * 180,
    ax,
    colors="tab:orange",
    levels=3,
    linewidths=3,
)
density_contour(
    np.log10(P_x_young_sim_all),
    chi_x_young_sim_all / np.pi * 180,
    ax,
    colors="tab:pink",
    levels=3,
    linewidths=3,
)

# Collect existing legend entries (from scatter plots, etc.).
handles, labels = ax.get_legend_handles_labels()

# Final legend with all items.
# Create proxy handles for legend.
kde_radio_proxy = mlines.Line2D(
    [], [], color="darkgray", linewidth=2, label="Radio KDE"
)
kde_x_proxy = mlines.Line2D(
    [], [], color="#d6c2ea", linewidth=2, label="X-ray KDE"
)
kde_xdins_proxy = mlines.Line2D(
    [], [], color="#ffb36b", linewidth=2, label="XDINS KDE"
)
kde_xyoung_proxy = mlines.Line2D(
    [], [], color="#FFB3D9", linewidth=2, label="X-ray young KDE"
)

# Add the KDE proxies.
handles.extend(
    [kde_radio_proxy, kde_x_proxy, kde_xdins_proxy, kde_xyoung_proxy]
)
labels.extend(
    [
        "Simulated radio KDE",
        "Simulated X-ray KDE",
        "Simulated XDINS KDE",
        "Simulated X-ray young KDE",
    ]
)

ax.legend(handles, labels, frameon=True, loc=0, prop={"size": 14})

plt.grid()

fig.savefig("plots/inclination_angle.pdf", bbox_inches="tight")